In [1]:
import math
from collections import Counter
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv("/BTC-USD.csv")

print(df.head())

         Date        Open        High         Low       Close   Adj Close  \
0  2014-09-17  465.864014  468.174011  452.421997  457.334015  457.334015   
1  2014-09-18  456.859985  456.859985  413.104004  424.440002  424.440002   
2  2014-09-19  424.102997  427.834991  384.532013  394.795990  394.795990   
3  2014-09-20  394.673004  423.295990  389.882996  408.903992  408.903992   
4  2014-09-21  408.084991  412.425995  393.181000  398.821014  398.821014   

     Volume  
0  21056800  
1  34483200  
2  37919700  
3  36863600  
4  26580100  


In [3]:
df["Movement"] = df["Close"].shift(-1) > df["Close"]

df["Movement"] = df["Movement"].map({True: "Up", False: "Down"})

df = df.dropna()

In [4]:
df["Open"] = pd.qcut(df["Open"], 3, labels=["Low", "Medium", "High"])

df["High"] = pd.qcut(df["High"], 3, labels=["Low", "Medium", "High"])

df["Low"] = pd.qcut(df["Low"], 3, labels=["Low", "Medium", "High"])

df["Volume"] = pd.qcut(df["Volume"], 3, labels=["Low", "Medium", "High"])

In [5]:
features = ["Open", "High", "Low", "Volume"]

target = "Movement"

In [6]:
def entropy(labels):

    total = len(labels)

    count = Counter(labels)

    ent = 0

    for value in count.values():

        probability = value / total

        ent -= probability * math.log2(probability)

    return ent

In [7]:
def information_gain(data, column):

    total_entropy = entropy(data[target])

    total_rows = len(data)

    weighted_entropy = 0

    for value, subset in data.groupby(column):

        weight = len(subset) / total_rows

        weighted_entropy += weight * entropy(subset[target])

    return total_entropy - weighted_entropy

In [11]:
def id3(data, features):

    labels = list(data[target])

    if len(labels) == 0:
        return "Unknown"

    if len(set(labels)) == 1:
        return labels[0]

    if len(features) == 0:
        return Counter(labels).most_common(1)[0][0]

    gains = {}

    for feature in features:
        gains[feature] = information_gain(data, feature)

    best_feature = max(gains, key=gains.get)

    tree = {best_feature: {}}

    remaining_features = [f for f in features if f != best_feature]

    for value, subset in data.groupby(best_feature, observed=False):

        if len(subset) == 0:
            tree[best_feature][value] = Counter(labels).most_common(1)[0][0]
        else:
            tree[best_feature][value] = id3(subset, remaining_features)

    return tree

In [12]:
def print_tree(tree, space=""):

    if not isinstance(tree, dict):
        print(space + "-> " + str(tree))
        return

    root = list(tree.keys())[0]

    for value in tree[root]:

        print(space + root + " = " + str(value))

        print_tree(tree[root][value], space + "    ")

In [13]:
print("ID3 Decision Tree")

id3_tree = id3(df, features)

print_tree(id3_tree)

ID3 Decision Tree
Low = Low
    Open = Low
        High = Low
            Volume = Low
                -> Up
            Volume = Medium
                -> Up
            Volume = High
                -> Up
        High = Medium
            -> Down
        High = High
            -> Up
    Open = Medium
        -> Down
    Open = High
        -> Up
Low = Medium
    Volume = Low
        High = Low
            -> Up
        High = Medium
            Open = Low
                -> Up
            Open = Medium
                -> Up
            Open = High
                -> Up
        High = High
            -> Up
    Volume = Medium
        Open = Low
            -> Down
        Open = Medium
            High = Low
                -> Up
            High = Medium
                -> Up
            High = High
                -> Down
        Open = High
            High = Low
                -> Up
            High = Medium
                -> Up
            High = High
                -> Down


/tmp/ipykernel_512/2343842883.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for value, subset in data.groupby(column):


In [14]:
encoded = df.copy()

for column in encoded.columns:

    encoder = LabelEncoder()

    encoded[column] = encoder.fit_transform(encoded[column])

In [15]:
X = encoded[features]

y = encoded[target]

In [16]:
c45 = DecisionTreeClassifier(
    criterion="entropy",
    random_state=0
)

c45.fit(X, y)

print("C4.5 Decision Tree")

print(export_text(c45, feature_names=features))

C4.5 Decision Tree
|--- High <= 0.50
|   |--- Volume <= 1.00
|   |   |--- Open <= 1.00
|   |   |   |--- Low <= 1.00
|   |   |   |   |--- class: 1
|   |   |   |--- Low >  1.00
|   |   |   |   |--- class: 0
|   |   |--- Open >  1.00
|   |   |   |--- Low <= 1.00
|   |   |   |   |--- class: 1
|   |   |   |--- Low >  1.00
|   |   |   |   |--- class: 1
|   |--- Volume >  1.00
|   |   |--- Open <= 1.00
|   |   |   |--- Low <= 1.00
|   |   |   |   |--- class: 1
|   |   |   |--- Low >  1.00
|   |   |   |   |--- class: 0
|   |   |--- Open >  1.00
|   |   |   |--- Low <= 1.00
|   |   |   |   |--- class: 0
|   |   |   |--- Low >  1.00
|   |   |   |   |--- class: 0
|--- High >  0.50
|   |--- Volume <= 0.50
|   |   |--- Low <= 1.00
|   |   |   |--- Open <= 1.00
|   |   |   |   |--- class: 0
|   |   |   |--- Open >  1.00
|   |   |   |   |--- class: 0
|   |   |--- Low >  1.00
|   |   |   |--- Open <= 1.00
|   |   |   |   |--- class: 1
|   |   |   |--- Open >  1.00
|   |   |   |   |--- class: 1
|   |--

In [17]:
cart = DecisionTreeClassifier(
    criterion="gini",
    random_state=0
)

cart.fit(X, y)

print("CART Decision Tree")

print(export_text(cart, feature_names=features))

CART Decision Tree
|--- High <= 0.50
|   |--- Volume <= 1.00
|   |   |--- Open <= 1.00
|   |   |   |--- Low <= 1.00
|   |   |   |   |--- class: 1
|   |   |   |--- Low >  1.00
|   |   |   |   |--- class: 0
|   |   |--- Open >  1.00
|   |   |   |--- Low <= 1.00
|   |   |   |   |--- class: 1
|   |   |   |--- Low >  1.00
|   |   |   |   |--- class: 1
|   |--- Volume >  1.00
|   |   |--- Open <= 1.00
|   |   |   |--- Low <= 1.00
|   |   |   |   |--- class: 1
|   |   |   |--- Low >  1.00
|   |   |   |   |--- class: 0
|   |   |--- Open >  1.00
|   |   |   |--- Low <= 1.00
|   |   |   |   |--- class: 0
|   |   |   |--- Low >  1.00
|   |   |   |   |--- class: 0
|--- High >  0.50
|   |--- Volume <= 0.50
|   |   |--- Low <= 1.00
|   |   |   |--- Open <= 1.00
|   |   |   |   |--- class: 0
|   |   |   |--- Open >  1.00
|   |   |   |   |--- class: 0
|   |   |--- Low >  1.00
|   |   |   |--- Open <= 1.00
|   |   |   |   |--- class: 1
|   |   |   |--- Open >  1.00
|   |   |   |   |--- class: 1
|   |--